In [22]:
from seleniumbase import Driver
import json
from selenium.webdriver.common.by import By
import pandas as pd
import time
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException, TimeoutException

In [23]:
driver = Driver(uc=True)   # undetected-chromedriver

In [14]:
BASE_URL = 'https://www.elmenus.com/cairo/delivery/downtown/features-order-online'

In [ ]:
#static
# urls = {
#         'Karam El Sham':'https://www.elmenus.com/alexandria/karam-el-sham-zmv2k/semouha-zm222/reviews',
#         "City Broast":'https://www.elmenus.com/alexandria/city-broast-ywx43/semouha-k79r4/reviews',
#         'Pizza King':'https://www.elmenus.com/alexandria/pizza-king-8vpk/al-raml-station-ommd4/reviews',
#         'dipndip':'https://www.elmenus.com/alexandria/dipndip-95pzm/sidi-gaber-zmnvm/reviews',
#         'Sparta Restaurant':'https://www.elmenus.com/alexandria/sparta-restaurant-o445/semouha-vyyl7/reviews'}

In [15]:
driver.get(BASE_URL)

MaxRetryError: HTTPConnectionPool(host='localhost', port=38861): Max retries exceeded with url: /session/0e849195ec07a70852f3d39f6ceab8fb/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=38861): Failed to establish a new connection: [Errno 111] Connection refused"))

In [ ]:
#old one
# elements = driver.find_elements(By.XPATH, '//h3[@class="card-title"]/a')

# for el in elements:
#     title = el.text
#     url = el.get_attribute("href")
#     print(title, url)


In [ ]:
# elements2 = driver.find_elements(By.XPATH, '//h3[contains(@class,"card-title")]/a')
# for e in elements2:
#     print(e.text)


In [37]:
def pressLoad(driver):
    try:
        load_more_btn = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable(
                (By.CSS_SELECTOR, "button.btn.btn-primary.btn--load-more")
            )
        )

        driver.execute_script("arguments[0].scrollIntoView(true);", load_more_btn)
        time.sleep(1)
        driver.execute_script("arguments[0].click();", load_more_btn)
        time.sleep(2)
    except (NoSuchElementException, TimeoutException, ElementClickInterceptedException):
        print("No more Load More button. Finished loading all restaurants.")
        return False

In [ ]:
# wait = WebDriverWait(driver, 10)

# while True:
#     try:
#         load_more_btn = wait.until(
#             EC.element_to_be_clickable(
#                 (By.CSS_SELECTOR, "button.load-more-btn")
#             )
#         )

#         driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", load_more_btn)
#         time.sleep(1)

#         driver.execute_script("arguments[0].click();", load_more_btn)
#         print("Clicked Load More")

#         time.sleep(3)  # wait for new restaurants to load

#     except (TimeoutException, ElementClickInterceptedException):
#         print("No more Load More button. Finished.")
#         break


# # Now scrape titles & urls
# elements = wait.until(EC.presence_of_all_elements_located(
#     (By.XPATH, '//h3[contains(@class,"card-title")]/a')
# ))
# urls = {}
# for el in elements:
#     title = el.text.strip()
#     url = el.get_attribute("href")
#     urls[title] = url+'/reviews'


Clicked Load More
Clicked Load More
Clicked Load More
Clicked Load More
Clicked Load More
No more Load More button. Finished.


In [ ]:
# for key,value in urls.items():
#     urls[key] = value + '/reviews'

In [24]:
with open('urls.json','r') as f:
    urls = json.load(f)

In [25]:
len(urls)

19

In [ ]:
# reviews_data = []
# seen_comments = set()

# comments = driver.find_elements(By.XPATH, '//div[contains(@class,"review__comment")]//p')

# for c in comments:
#     try:
#         text = c.text.strip()
#     except:
#         continue    
#     if not text:
#         continue

#     if text in seen_comments:
#         continue

#     seen_comments.add(text)
#     reviews_data.append({"comment": text})

In [ ]:
# len(reviews_data)

443

In [ ]:
def extract_elmenu_reviews(driver, url, max_reviews=3000):

    driver.get(url)
    time.sleep(3)

    reviews_data = []
    seen_comments = set()
    old_len = 0
    count = 0
    while True:
        # wait for reviews to be present
        WebDriverWait(driver, 2).until(
            EC.presence_of_all_elements_located(
                (By.XPATH, '//div[contains(@class,"review__comment")]//p')
            )
        )

        comments = driver.find_elements(By.XPATH, '//div[contains(@class,"review__comment")]//p')

        for c in comments:
            try:
                text = c.text.strip()
            except:
                continue
            if not text:
                continue

            if text in seen_comments:
                continue

            seen_comments.add(text)
            reviews_data.append({"comment": text})

        # print(f"Collected {len(reviews_data)} comments")
        if len(reviews_data) == old_len:
            count+=1
            
        if len(reviews_data) == old_len and count>=3:
            break
        if len(reviews_data) >= max_reviews:
            break
        old_len = len(reviews_data)
        # try clicking Load More button
        try:
            load_more_btn = WebDriverWait(driver, 1).until(
                EC.element_to_be_clickable(
                    (By.CSS_SELECTOR, 'button.btn.btn-primary.btn--load-more')
                )
            )
            driver.execute_script("arguments[0].scrollIntoView(true);", load_more_btn)
            time.sleep(1)
            driver.execute_script("arguments[0].click();", load_more_btn)
            time.sleep(1.2)

        except (NoSuchElementException, ElementClickInterceptedException):
            print("No more Load More button. Finished.")
            break

    df = pd.DataFrame(reviews_data)
    return df


In [ ]:

# Open a file in write mode and save the dictionary as JSON
with open("urls.json", "w") as json_file:
    json.dump(urls, json_file, indent=4) 


In [26]:
skip = 0
for key,value in urls.items():
    if skip < 0:
        skip+=1
        print(f'skip {key}')
        continue
    df = extract_elmenu_reviews(driver,value)
    df.to_csv(f'elmenu_{key}_resturant.csv')
    print(f'{key} has been scraped we got {df.shape}')

TimeoutException: Message: 
Stacktrace:
#0 0x579a272d508a <unknown>
#1 0x579a26d74a70 <unknown>
#2 0x579a26dc6907 <unknown>
#3 0x579a26dc6b01 <unknown>
#4 0x579a26e14d54 <unknown>
#5 0x579a26dec40d <unknown>
#6 0x579a26e1214f <unknown>
#7 0x579a26dec1b3 <unknown>
#8 0x579a26db859b <unknown>
#9 0x579a26db9971 <unknown>
#10 0x579a2729a25b <unknown>
#11 0x579a2729dfa9 <unknown>
#12 0x579a27281339 <unknown>
#13 0x579a2729eb58 <unknown>
#14 0x579a27265c1f <unknown>
#15 0x579a272c2118 <unknown>
#16 0x579a272c22f6 <unknown>
#17 0x579a272d4066 <unknown>
#18 0x7a5acde9caa4 <unknown>
#19 0x7a5acdf29c6c <unknown>


In [18]:
# df.to_csv('elmenu__resturant.csv')